# TP Jour 2 — Modèle d'Exécution Spark, Jointures Distribuées & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze**, nettoyer les anomalies, typer fortement les colonnes (dates/timestamps, numériques) et créer vos colonnes métier dérivées.
3. Mettre en œuvre une **jointure distribuée optimisée** avec **Broadcast Hash Join** (`F.broadcast`) sur une table de référence (sans créer de produit cartésien N-M !).
4. Sauvegarder votre table analytique propre dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

Session Spark active — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze & Diagnostic Qualité

### Exercice 1.1 — Profiler les valeurs manquantes, doublons et plages de valeurs
1. Chargez votre (ou vos) table(s) Delta depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` par colonne clé ainsi que le nombre de doublons éventuels sur l'identifiant métier.

In [7]:
# TODO : Charger votre table principale depuis `/lakehouse/bronze/...`
BRONZE_MAIN_PATH = "/lakehouse/bronze/matches"

df_bronze = (
    spark.read
    .format("delta")
    .load(BRONZE_MAIN_PATH)
)

assert df_bronze is not None, "TODO : Chargez votre table Bronze principale dans `df_bronze`."

print(f"Nombre de lignes : {df_bronze.count():,}")
print("Schéma de la table Bronze :")
df_bronze.printSchema()

# TODO : Calculer et afficher le nombre de valeurs nulles sur vos colonnes principales
null_report = df_bronze.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in df_bronze.columns
])

assert null_report is not None, "TODO : Calculez le diagnostic de valeurs nulles `null_report`."

null_report.show(truncate=False)

Nombre de lignes : 238,858
Schéma de la table Bronze :
root
 |-- division: string (nullable = true)
 |-- matchdate: date (nullable = true)
 |-- matchtime: timestamp (nullable = true)
 |-- hometeam: string (nullable = true)
 |-- awayteam: string (nullable = true)
 |-- homeelo: double (nullable = true)
 |-- awayelo: double (nullable = true)
 |-- form3home: double (nullable = true)
 |-- form5home: double (nullable = true)
 |-- form3away: double (nullable = true)
 |-- form5away: double (nullable = true)
 |-- fthome: double (nullable = true)
 |-- ftaway: double (nullable = true)
 |-- ftresult: string (nullable = true)
 |-- hthome: double (nullable = true)
 |-- htaway: double (nullable = true)
 |-- htresult: string (nullable = true)
 |-- homeshots: double (nullable = true)
 |-- awayshots: double (nullable = true)
 |-- hometarget: double (nullable = true)
 |-- awaytarget: double (nullable = true)
 |-- homefouls: double (nullable = true)
 |-- awayfouls: double (nullable = true)
 |-- homecorner

In [8]:
# Diagnostic des doublons sur l'identifiant métier
business_key = [
    "matchdate",
    "division",
    "hometeam",
    "awayteam"
]

# Nombre de doublons sur cette combinaison
total_rows = df_bronze.count()

distinct_matches = (
    df_bronze
    .dropDuplicates(business_key)
    .count()
)

duplicate_count = total_rows - distinct_matches

print("DIAGNOSTIC DES DOUBLONS")
print(f"Nombre total de matchs       : {total_rows:,}")
print(f"Matchs uniques               : {distinct_matches:,}")
print(f"Doublons détectés            : {duplicate_count:,}")

DIAGNOSTIC DES DOUBLONS
Nombre total de matchs       : 238,858
Matchs uniques               : 238,858
Doublons détectés            : 0


## 2. Nettoyage, Typage Fort & Jointure Optimisée (`BroadcastHashJoin`)

### Rappels d'architecture distribuée :
- **Attention aux jointures N-à-M :** Si vous joignez deux tables de détail partageant une clé non unique, pré-agrégez d'abord à la bonne granularité pour ne pas dupliquer les lignes et fausser vos sommes !
- **Broadcast Hash Join (`F.broadcast(petite_table)`) :** Lorsqu'une table de dimension / nomenclature fait quelques kilo-octets ou méga-octets, la diffuser sur chaque exécuteur supprime totalement l'étape coûteuse de **Shuffle** sur la grande table de faits.
  *(Si votre dataset Kaggle ne comporte qu'un seul fichier plat, construisez ou extrayez une table de référence/catégories dédiée pour pratiquer cette jointure et vérifier le plan physique !)*

### Exercice 2.1 — Nettoyer, enrichir par Broadcast Join et écrire `/lakehouse/silver/<table_silver>`
1. Supprimez les doublons (`dropDuplicates`), filtrez les valeurs techniquement impossibles (ex. durées ou montants négatifs) et convertissez explicitement vos dates (`F.to_timestamp` / `F.to_date`).
2. Créez au moins **2 nouvelles colonnes métier dérivées** utiles pour vos analyses des Jours 3 & 4 (ex. durée écoulée, écart, ratio, tranche horaire, mois `yyyy-MM`, indicateur binaire `0/1`...).
3. Réalisez une jointure avec `F.broadcast(...)` et affichez `.explain("simple")` pour prouver la présence de `BroadcastHashJoin`.
4. Écrivez la table résultante au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [13]:

SILVER_MAIN_PATH = "/lakehouse/silver/matches"

# 1. Nettoyage et typage fort
business_key = ["matchdate", "division", "hometeam", "awayteam"]

df_clean = (
    df_bronze
    .dropDuplicates(business_key)
    # Suppression des matchs sans résultat ou sans score final
    .filter(
        F.col("matchdate").isNotNull()
        & F.col("hometeam").isNotNull()
        & F.col("awayteam").isNotNull()
        & F.col("fthome").isNotNull()
        & F.col("ftaway").isNotNull()
        & F.col("ftresult").isin("H", "D", "A")
    )
    # Suppression des scores techniquement impossibles
    .filter(
        (F.col("fthome") >= 0)
        & (F.col("ftaway") >= 0)
    )
    # Conversion explicite des dates et typage numérique
    .withColumn("matchdate", F.to_date("matchdate"))
    .withColumn("matchtime", F.to_timestamp("matchtime"))
    .withColumn("fthome", F.col("fthome").cast("double"))
    .withColumn("ftaway", F.col("ftaway").cast("double"))
    .withColumn("homeelo", F.col("homeelo").cast("double"))
    .withColumn("awayelo", F.col("awayelo").cast("double"))
)

# 2. Création de colonnes métier dérivées
df_enriched = (
    df_clean
    # Différence de buts entre les deux équipes
    .withColumn(
        "goal_difference",
        F.col("fthome") - F.col("ftaway")
    )
    # Différence de classement Elo
    .withColumn(
        "elo_difference",
        F.col("homeelo") - F.col("awayelo")
    )
    # Mois du match au format yyyy-MM
    .withColumn(
        "match_month",
        F.date_format("matchdate", "yyyy-MM")
    )
    # Indicateur binaire : 1 si victoire à domicile, 0 sinon
    .withColumn(
        "home_win",
        F.when(F.col("ftresult") == "H", 1).otherwise(0)
    )
)

# 3. Création d'une petite table de référence des résultats
result_dimension = spark.createDataFrame(
    [
        ("H", "Victoire domicile"),
        ("D", "Match nul"),
        ("A", "Victoire extérieur")
    ],
    ["ftresult", "result_label"]
)

# Jointure optimisée avec Broadcast
df_silver = df_enriched.join(
    F.broadcast(result_dimension),
    on="ftresult",
    how="left"
)

assert df_silver is not None, "TODO : Construisez le DataFrame nettoyé `df_silver`."

# Vérification du plan physique
df_silver.explain("simple")

# 4. Sauvegarde au format Delta
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_MAIN_PATH)
)

print(
    f"Table Silver écrite dans {SILVER_MAIN_PATH} : "
    f"{spark.read.format('delta').load(SILVER_MAIN_PATH).count():,} lignes"
)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [ftresult#9147, division#2583, matchdate#2584, matchtime#9129, hometeam#2586, awayteam#2587, homeelo#9131, awayelo#9133, form3home#9135, form5home#9137, form3away#9139, form5away#9141, fthome#9143, ftaway#9145, hthome#9149, htaway#9151, htresult#9153, homeshots#9155, awayshots#9157, hometarget#9159, awaytarget#9161, homefouls#9163, awayfouls#9165, homecorners#9167, ... 31 more fields]
   +- BroadcastHashJoin [ftresult#9147], [ftresult#9069], LeftOuter, BuildRight, false
      :- Project [division#2583, matchdate#2584, matchtime#9129, hometeam#2586, awayteam#2587, homeelo#9131, awayelo#9133, form3home#9135, form5home#9137, form3away#9139, form5away#9141, fthome#9143, ftaway#9145, ftresult#9147, hthome#9149, htaway#9151, htresult#9153, homeshots#9155, awayshots#9157, hometarget#9159, awaytarget#9161, homefouls#9163, awayfouls#9165, homecorners#9167, ... 30 more fields]
      :  +- Filter ((((isnotnull(fthome#9143) AND isn

26/11/17 13:01:08 WARN MemoryManager: Total allocation exceeds 95.00% (2,040,109,440 bytes) of heap memory
Scaling row group sizes to 95.00% for 16 writers


Table Silver écrite dans /lakehouse/silver/matches : 238,854 lignes


## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [ ]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# 2. TODO : Tester le Schema Enforcement avec un DataFrame au mauvais schéma dans un bloc try/except
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé — écriture invalide bloquée :", str(e).splitlines()[0])

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [ ]:
# TODO : Exécuter un MERGE INTO sur `SILVER_MAIN_PATH` puis comparer versionAsOf=0 et la version courante
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
...

history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)
assert history_df.count() >= 2, "TODO : Exécutez au moins une opération MERGE/UPDATE pour créer une version >= 1 !"